# Bab 15 · scikit-learn: Melatih dan Menilai dengan Jujur

**Notebook praktikum mahasiswa**  
Versi 2.0 · Pemrograman Komputer

- Membagi data sebelum mempelajari prapemrosesan.
- Membandingkan model dengan patokan menggunakan validasi yang sama.
- Menafsirkan precision, recall, dan F1 untuk kelas minoritas.

### Petunjuk menjalankan sel · versi 2.0

Jalankan sel berurutan dari atas ke bawah. Setiap fungsi mandiri diletakkan pada sel tersendiri; sel pemanggilan atau pengujiannya menyusul setelah definisi. Setelah menyunting fungsi, jalankan ulang sel definisinya, lalu sel pengujiannya.

Sel persiapan dan fungsi pemeriksa cukup dijalankan; bagian yang Anda kerjakan ditandai **[ISI KODE]**. Metode yang membentuk satu kelas serta fungsi bersarang tetap disatukan karena merupakan satu kesatuan Python.

## Alur praktikum

**Duga → Jalankan → Selidiki → Isi kode → Periksa → Jelaskan**

Perkiraan waktu: 90–120 menit. Kerjakan berpasangan; tukar peran penulis kode dan pemeriksa setiap dua latihan.

| Penanda | Yang Anda kerjakan |
|---|---|
| [BACA] | Pahami konsep, kontrak fungsi, dan kasus batas. |
| [DUGA] | Tulis prediksi sebelum menjalankan contoh. |
| [COBA] | Jalankan contoh dan ubah satu hal untuk menyelidiki hasilnya. |
| [ISI KODE] | Lengkapi fungsi atau kelas; pertahankan nama dan parameternya. |
| [CEK OTOMATIS] | Jalankan pengujian yang terlihat, lalu gunakan pesannya untuk memperbaiki kode. |
| [REFLEKSI] | Jelaskan alasan dan bukti, bukan hanya menyalin keluaran. |

Impor file `.ipynb` ini ke notebook Python di Kaggle. Gunakan CPU; data kecil disediakan dalam notebook. Jalankan sel dari atas ke bawah. Pustaka yang diperlukan diimpor pada sel persiapan; tidak ada perintah instalasi atau unduhan.

`BELUM DIISI` adalah status normal pada notebook awal. Ganti `raise BelumDiisi()` dengan pekerjaan Anda. `LULUS` berarti memenuhi kasus uji yang tersedia, bukan bukti bahwa semua kemungkinan input sudah benar. Sel pengujian harus tetap utuh.

Jika kode berulang tanpa selesai, hentikan eksekusi, periksa batas perulangan, lalu jalankan ulang. Sebelum mengumpulkan, mulai ulang sesi Python dan jalankan seluruh sel agar hasil tidak bergantung pada variabel lama.

### Identitas

- Nama: …
- NIM: …
- Rekan diskusi: …
- Tanggal: …

**Persiapan dan pengaturan** · bagian 1 dari 10

In [ ]:
# [COBA] Jalankan sekali di awal; pemeriksaan tersedia untuk dibaca.
import math
import sys
from copy import deepcopy
from pathlib import Path
from tempfile import TemporaryDirectory

**Definisi `BelumDiisi`** · bagian 2 dari 10

In [ ]:
class BelumDiisi(Exception):
    """Penanda latihan yang belum dikerjakan."""

**Definisi `sama`** · bagian 3 dari 10

In [ ]:
def sama(aktual, harapan):
    assert aktual == harapan, f"Diharapkan {harapan!r}; diperoleh {aktual!r}"

**Definisi `dekat`** · bagian 4 dari 10

In [ ]:
def dekat(aktual, harapan, atol=1e-8, rtol=1e-7):
    assert math.isclose(
        aktual, harapan, abs_tol=atol, rel_tol=rtol
    ), f"Diharapkan sekitar {harapan!r}; diperoleh {aktual!r}"

**Definisi `harus_galat`** · bagian 5 dari 10

In [ ]:
def harus_galat(jenis, panggil):
    try:
        panggil()
    except BelumDiisi:
        raise
    except jenis:
        return
    raise AssertionError(f"Seharusnya memunculkan {jenis.__name__}")

**Persiapan dan pengaturan** · bagian 6 dari 10

In [ ]:
DAFTAR_UJI = {}

**Definisi `cek`** · bagian 7 dari 10

In [ ]:
def cek(nomor, fungsi_uji, tampil=True):
    DAFTAR_UJI[nomor] = fungsi_uji
    try:
        fungsi_uji()
        status, pesan = "LULUS", "Semua kasus uji pada latihan ini sesuai."
    except BelumDiisi:
        status, pesan = (
            "BELUM DIISI",
            "Lengkapi sel [ISI KODE], jalankan, lalu ulangi pemeriksaan.",
        )
    except AssertionError as err:
        status, pesan = (
            "PERLU PERBAIKAN",
            str(err) or "Hasil belum sesuai kontrak latihan.",
        )
    except Exception as err:
        status, pesan = "GALAT", f"{type(err).__name__}: {err}"
    if tampil:
        print(f"Latihan {nomor} | {status}\n{pesan}")
    return status

**Definisi `rekap`** · bagian 8 dari 10

In [ ]:
def rekap():
    # Uji ulang fungsi terkini agar rekap tidak memakai status lama.
    hasil = {
        nomor: cek(nomor, uji, tampil=False)
        for nomor, uji in sorted(DAFTAR_UJI.items())
    }
    for nomor, status in hasil.items():
        print(f"  Latihan {nomor}: {status}")
    lulus = sum(s == "LULUS" for s in hasil.values())
    print(f"\nKemajuan uji otomatis: {lulus}/{JUMLAH_LATIHAN} latihan lulus.")
    print(
        "Refleksi, penjelasan, dan kualitas penyajian diperiksa bersama asisten."
    )
    return hasil

**Persiapan dan pengaturan** · bagian 9 dari 10

In [ ]:
print("Python:", sys.version.split()[0])
print("Siap. Jalankan notebook dari atas ke bawah.")
JUMLAH_LATIHAN = 4
import numpy as np

print("NumPy:", np.__version__)
import pandas as pd

print("pandas:", pd.__version__)
import sklearn
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split, KFold, cross_val_score

**Persiapan dan pengaturan** · bagian 10 dari 10

In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.compose import ColumnTransformer

print("scikit-learn:", sklearn.__version__)

## [BACA] Konsep inti

Pipeline menjaga prapemrosesan tetap bersama model. Dalam validasi silang, setiap fold harus mempelajari imputasi, skala, dan pemilihan fitur hanya dari bagian latihnya. Tetapkan metrik sebelum membandingkan model. Akurasi tinggi pada kelas timpang dapat menutupi kegagalan menangkap kelas positif. Skor validasi silang bukan pengganti evaluasi akhir pada holdout yang tidak dipakai untuk memilih model.

## [DUGA] Prediksi sebelum eksekusi

Apakah akurasi 90% sudah memadai bila kelas positif adalah kejadian yang harus ditemukan?

**Prediksi saya:** …

**Alasan:** …

In [ ]:
# [COBA]
y = np.array([0] * 18 + [1] * 2)
p = np.zeros_like(y)
print("akurasi:", np.mean(y == p))
print("recall positif:", recall_score(y, p, zero_division=0))
print("F1 positif:", f1_score(y, p, zero_division=0))

**[REFLEKSI]** Apa perbedaan prediksi dan hasil? Ubah satu input pada contoh, tulis hasilnya, lalu jelaskan konsep yang ditunjukkan.

**Jawaban:** …

## Latihan 1 · Split yang dapat diulang

**[ISI KODE]**

Buat `bagi_klasifikasi(X, y, seed=42)` yang mengembalikan `(X_latih, X_uji, y_latih, y_uji)`. Gunakan train_test_split dengan test_size=0.25, stratify=y, dan random_state=seed. X ndarray 2D, y label biner, jumlah per kelas mencukupi. Data pada bab ini dianggap pengamatan independen, bukan deret waktu.

> Petunjuk: Teruskan argumen stratify dan random_state kepada train_test_split.

In [ ]:
# [ISI KODE]
def bagi_klasifikasi(X, y, seed=42):
    raise BelumDiisi()

**Definisi `uji_01`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_01():
    X = np.arange(80).reshape(40, 2)
    y = np.array([0] * 32 + [1] * 8)
    a = bagi_klasifikasi(X, y, 42)
    b = bagi_klasifikasi(X, y, 42)
    sama([len(v) for v in a], [30, 10, 30, 10])
    for u, v in zip(a, b):
        np.testing.assert_array_equal(u, v)
    sama(int(a[3].sum()), 2)
    assert set(a[0][:, 0]).isdisjoint(
        set(a[1][:, 0])
    ), "Latih dan uji bertumpang tindih."
    sama(set(a[0][:, 0]) | set(a[1][:, 0]), set(X[:, 0]))
    for baris, label in zip(a[1], a[3]):
        sama(int(label), int(y[baris[0] // 2]))
    c = bagi_klasifikasi(X, y, 19)
    assert not np.array_equal(a[1], c[1]), "Parameter seed harus digunakan."

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(1, uji_01)

## Latihan 2 · Pipeline prapemrosesan dan Ridge

**[ISI KODE]**

Buat `buat_pipeline(alpha=1.0)` yang mengembalikan Pipeline belum dilatih dengan langkah bernama `imputer`, `skala`, `model`. Gunakan SimpleImputer(strategy="median"), StandardScaler, dan Ridge(alpha=alpha). Masukan numerik bisa memuat NaN; tidak ada kolom seluruhnya hilang.

> Petunjuk: Pipeline menerima list pasangan (nama_langkah, objek_transformer_atau_model).

In [ ]:
# [ISI KODE]
def buat_pipeline(alpha=1.0):
    raise BelumDiisi()

**Definisi `uji_02`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_02():
    p = buat_pipeline(2.5)
    assert isinstance(p, Pipeline), "Kembalikan Pipeline."
    sama(list(p.named_steps), ["imputer", "skala", "model"])
    assert (
        isinstance(p["imputer"], SimpleImputer)
        and p["imputer"].strategy == "median"
    )
    assert isinstance(p["skala"], StandardScaler)
    assert isinstance(p["model"], Ridge)
    dekat(p["model"].alpha, 2.5)
    assert not hasattr(
        p["skala"], "mean_"
    ), "Pipeline awal belum boleh dilatih."
    X = np.array([[1.0, 1.0], [3.0, np.nan], [5.0, 3.0], [7.0, 5.0]])
    y = np.array([1.0, 2.0, 3.0, 4.0])
    p.fit(X, y)
    np.testing.assert_allclose(p["imputer"].statistics_, [4, 3])
    np.testing.assert_allclose(p["skala"].mean_, [4, 3])
    awal = p["skala"].mean_.copy()
    assert np.isfinite(p.predict([[999.0, np.nan]])).all()
    np.testing.assert_array_equal(p["skala"].mean_, awal)

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(2, uji_02)

## Latihan 3 · Validasi silang dengan metrik MAE

**[ISI KODE]**

Buat `nilai_cv(model, X, y)` untuk regresi dengan minimal 10 baris independen. Gunakan KFold(n_splits=5, shuffle=True, random_state=42) dan cross_val_score(scoring="neg_mean_absolute_error"). Kembalikan dict `mae_fold` (array 5 nilai MAE positif), `rerata`, `sd` populasi. Jangan melatih model asal secara langsung; cross_val_score akan mengklonanya.

> Petunjuk: scikit-learn memakai skor yang makin besar makin baik; neg_MAE perlu dikalikan -1.

In [ ]:
# [ISI KODE]
def nilai_cv(model, X, y):
    raise BelumDiisi()

**Definisi `uji_03`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_03():
    X = np.arange(20, dtype=float).reshape(-1, 1)
    y = 2 * X[:, 0] + 1
    model = DummyRegressor(strategy="median")
    h = nilai_cv(model, X, y)
    assert isinstance(h["mae_fold"], np.ndarray) and h["mae_fold"].shape == (
        5,
    )
    # Referensi evaluasi patokan dihitung dari split yang sama, bukan dari jawaban latihan.
    harapan = []
    for ilat, iuji in KFold(5, shuffle=True, random_state=42).split(X):
        harapan.append(np.mean(np.abs(y[iuji] - np.median(y[ilat]))))
    np.testing.assert_allclose(h["mae_fold"], harapan)
    dekat(h["rerata"], np.mean(harapan))
    dekat(h["sd"], np.std(harapan))
    assert not hasattr(model, "constant_"), "Model asal tidak boleh dilatih."
    print("Patokan MAE CV:", h)

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(3, uji_03)

## Latihan 4 · Metrik kelas positif

**[ISI KODE]**

Buat `metrik_biner(y, pred)` untuk label 0/1. Kembalikan dict `precision`, `recall`, `f1` kelas positif 1 dengan zero_division=0. Kembalikan juga `tp`, `fp`, `fn` sebagai integer. Kedua input 1D sama panjang dan tidak kosong.

> Petunjuk: Pisahkan hasil positif benar, positif palsu, dan positif yang terlewat.

In [ ]:
# [ISI KODE]
def metrik_biner(y, pred):
    raise BelumDiisi()

**Definisi `uji_04`** · bagian 1 dari 2

In [ ]:
# [CEK OTOMATIS] Kasus uji terbuka; jalankan setelah sel jawaban.
def uji_04():
    h = metrik_biner([1, 1, 1, 0, 0], [1, 0, 0, 1, 0])
    sama((h["tp"], h["fp"], h["fn"]), (1, 1, 2))
    dekat(h["precision"], 0.5)
    dekat(h["recall"], 1 / 3)
    dekat(h["f1"], 0.4)
    z = metrik_biner([0, 0, 1], [0, 0, 0])
    sama((z["precision"], z["recall"], z["f1"]), (0, 0, 0))

**Jalankan pemeriksaan** · bagian 2 dari 2

In [ ]:
cek(4, uji_04)

## [REFLEKSI] Refleksi akhir

Bagian ini membantu Anda merangkum pemahaman, mengenali kesulitan, dan menjelaskan alasan di balik kode. Tulis jawaban singkat berdasarkan percobaan Anda; bukan sekadar menyalin keluaran.

1. Pilih satu latihan. Jelaskan alur kode Anda dengan satu contoh input dan hasilnya.
2. Tuliskan satu kesalahan yang sempat terjadi, penyebabnya, dan cara memperbaikinya.
3. Usulkan satu kasus uji tambahan yang belum tercakup. Nyatakan hasil yang Anda harapkan dan alasannya.
4. Apa batas kesimpulan yang boleh dibuat dari hasil praktikum ini?

**Jawaban:** …

### Tantangan pengembangan

Tambahkan kasus uji usulan Anda pada sel di bawah. Pastikan kasus tersebut bisa membedakan implementasi benar dan satu kesalahan yang masuk akal. Diskusikan dengan asisten sebelum mengubah kontrak fungsi.

In [ ]:
# [ISI KODE OPSIONAL] Tambahkan eksperimen atau pengujian buatan Anda.
# Jelaskan harapan Anda pada komentar sebelum menjalankannya.

In [ ]:
# [CEK OTOMATIS] Uji ulang seluruh latihan yang sudah didaftarkan.
status_akhir = rekap()

## Sebelum mengumpulkan

- [ ] Identitas dan prediksi sudah diisi.
- [ ] Semua latihan sudah dikerjakan dan diperiksa dari sesi baru.
- [ ] refleksi akhir berisi penjelasan dengan bukti keluaran.
- [ ] Notebook disimpan dengan nama dan NIM; jangan hanya mengumpulkan HTML.

Rubrik diskusi: ketepatan kode 60%, penjelasan dan kasus batas 25%, keterbacaan serta kemampuan dijalankan ulang 15%. Rekap otomatis membantu belajar; penilaian akhir tetap memerlukan pemeriksaan asisten.

Rujukan: bab yang bersesuaian pada buku *Python untuk Machine Learning dan Data Science* dan modul praktikum. Latihan di notebook ini merupakan adaptasi terarah untuk praktikum, bukan seluruh soal akhir bab.